# ML · Encuentro 1 · Datos y primera predicción

Del problema a la primera predicción

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [Primero la pregunta, después el modelo](#scrollTo=tema-ml-problema)
2. [Un registro vacío no es cero](#scrollTo=tema-ml-preparacion)
3. [Una predicción también tiene error](#scrollTo=tema-ml-regresion)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · Definir entradas y respuesta

**Lo que vas a poder hacer:** Diseñar una primera predicción útil para planear una operación.

Una cooperativa necesita anticipar producción para organizar transporte antes de cosechar.

### Entiende la idea

El aprendizaje automático permite construir una regla a partir de ejemplos y comprobarla en otros casos. En esta primera tarea queremos estimar rendimiento para planear transporte. El recorrido tiene tres momentos distintos: entrenar ajusta la regla con parejas de entradas y respuestas; predecir calcula una salida para otro caso; evaluar compara esa salida con la respuesta observada. Aprender una relación no demuestra que una variable cause la otra.

### Un caso pequeño antes del programa

**Anticipar un lote reservado**

Caso resuelto con datos inventados del mismo periodo: lluvia en milímetros (mm) y rendimiento en toneladas por hectárea (t/ha). Las primeras cuatro filas sirven para ajustar la regla. B quedó fuera de ese ajuste; aquí mostramos su respuesta conocida para comprobar el resultado.

| Uso | Lluvia (mm) | Rendimiento (t/ha) |
| --- | --- | --- |
| Historia 1 | 400 | 2,00 |
| Historia 2 | 500 | 2,50 |
| Historia 3 | 600 | 3,00 |
| Historia 4 | 700 | 3,50 |
| Prueba B: respuesta revelada | 550 | 3,10 |

1. La entrada es lluvia; la respuesta buscada es rendimiento. Cada fila representa un lote en un periodo.
2. El patrón histórico suma 0,50 t/ha por cada 100 mm: para 550 mm predice 2,75 t/ha.
3. Al revelar B, comparamos 3,10 con 2,75: el error absoluto es 0,35 t/ha. Predecir no es adivinar exactamente.

**Qué demuestra:** Para B estimamos 2,75 t/ha y observamos 3,10 t/ha: el error es 0,35 t/ha. Si B tiene 10 ha, la estimación total sería 27,5 toneladas y la producción observada 31: una diferencia de 3,5 toneladas para revisar al planear transporte. Esta relación inventada no demuestra que aumentar lluvia cause ese aumento.

### Ahora identifica los datos de esta práctica

Cada fila es una finca simulada. Lluvia en mm, temperatura en °C y fertilizante en kg/ha son entradas; rendimiento en t/ha es la respuesta.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Son 240 fincas simuladas, distintas de las cuatro parejas del ejemplo pequeño del aula. La regla que genera los datos no es una recomendación agronómica.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


### 1.1. Crear la tabla y su respuesta conocida

Generamos lluvia entre 200 y 800 mm, temperatura entre 16 y 30 °C y fertilizante entre 50 y 180 kg/ha. Son rangos elegidos para la simulación, no límites agronómicos generales.

El rendimiento **también se inventa** con una regla: aumenta con lluvia y fertilizante; disminuye al alejarse de 23 °C; lleva una pequeña variación aleatoria. Los números de esa regla son decisiones didácticas. Así conocemos una respuesta para cada fila y podemos comprobar después las predicciones.

`rng.uniform` genera las entradas dentro de los rangos. `rng.normal` añade variación. `**2` eleva al cuadrado la diferencia de temperatura: crea una relación curva. El modelo de regresión aprenderá de las filas; no recibe esta regla generadora.

**Lee la salida:** cada fila muestra las tres entradas y el rendimiento simulado correspondiente.


In [ ]:
n = 240
X = pd.DataFrame({
    'lluvia_mm': rng.uniform(200, 800, n),
    'temperatura_c': rng.uniform(16, 30, n),
    'fertilizante_kg_ha': rng.uniform(50, 180, n)
})
y = (1 + .004 * X.lluvia_mm + .015 * X.fertilizante_kg_ha
     - .03 * (X.temperatura_c - 23)**2 + rng.normal(0, .25, n))
display(X.assign(rendimiento_t_ha=y).head())


### Lee, comprueba y explica

Cada fila es una finca simulada. Lluvia en mm, temperatura en °C y fertilizante en kg/ha son entradas; rendimiento en t/ha es la respuesta.

**Prueba pequeña:** Son 240 fincas simuladas, distintas de las cuatro parejas del ejemplo pequeño del aula. La regla que genera los datos no es una recomendación agronómica.

**Vuelve al caso:** Define la unidad de cada fila, las entradas disponibles al decidir y la respuesta que quieres estimar. Comprueba las predicciones con casos que quedaron fuera del ajuste.

**Pregunta:** ¿Qué distingue entrenar, predecir y comprobar una predicción?

**Respuesta razonada:** Entrenar ajusta la regla con ejemplos conocidos; predecir calcula una salida para otro caso; comprobar compara esa salida con su respuesta observada. La respuesta de prueba no se usa para ajustar.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-problema)


## Tema 2 · Preparar y reservar antes de ajustar

**Lo que vas a poder hacer:** Preparar una tabla con datos ausentes sin contaminar la prueba.

Si falla una estación, escribir cero confundiría ausencia de medida con ausencia de lluvia.

### Entiende la idea

Una regla solo puede interpretarse si sus entradas conservan un significado consistente. Preparar datos implica revisar unidades, registros y ausencias, y aplicar las transformaciones que requiera el método. Aquí estudiamos una ausencia de lluvia: la completamos con una estimación calculada en entrenamiento y dejamos constancia de que no fue medida. La mediana es una opción para este ejemplo; antes de usarla hay que investigar por qué falta el dato.

### Un caso pequeño antes del programa

**La estación B no registró lluvia**

Cuatro fincas de entrenamiento y lluvia inventada del mismo periodo. B tiene una ausencia, no una medida de cero.

| Finca | Registro de lluvia (mm) | Tipo de valor |
| --- | --- | --- |
| A | 500 | Medido |
| B | Sin registro | Ausente |
| C | 600 | Medido |
| D | 700 | Medido |

1. Tomamos los registros conocidos de entrenamiento y los ordenamos: 500, 600, 700 mm.
2. El valor central es 600 mm. Usamos esa mediana para completar B y conservamos la marca de estimación.
3. Si faltara lluvia en prueba, aplicaríamos esos 600 mm, sin recalcular usando la prueba.

**Qué demuestra:** La fila queda utilizable mediante una regla explícita; seguimos sin saber cuánto llovió realmente en B.

### Ahora identifica los datos de esta práctica

Veinte registros de lluvia están ausentes. Se reservan 60 fincas y la mediana se aprende únicamente con las 180 de entrenamiento.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Imputar no recupera una medición real. La misma mediana aprendida se aplica después a la prueba.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


### 1.1. Crear la tabla y su respuesta conocida

Generamos lluvia entre 200 y 800 mm, temperatura entre 16 y 30 °C y fertilizante entre 50 y 180 kg/ha. Son rangos elegidos para la simulación, no límites agronómicos generales.

El rendimiento **también se inventa** con una regla: aumenta con lluvia y fertilizante; disminuye al alejarse de 23 °C; lleva una pequeña variación aleatoria. Los números de esa regla son decisiones didácticas. Así conocemos una respuesta para cada fila y podemos comprobar después las predicciones.

`rng.uniform` genera las entradas dentro de los rangos. `rng.normal` añade variación. `**2` eleva al cuadrado la diferencia de temperatura: crea una relación curva. El modelo de regresión aprenderá de las filas; no recibe esta regla generadora.

**Lee la salida:** cada fila muestra las tres entradas y el rendimiento simulado correspondiente.


In [ ]:
n = 240
X = pd.DataFrame({
    'lluvia_mm': rng.uniform(200, 800, n),
    'temperatura_c': rng.uniform(16, 30, n),
    'fertilizante_kg_ha': rng.uniform(50, 180, n)
})
y = (1 + .004 * X.lluvia_mm + .015 * X.fertilizante_kg_ha
     - .03 * (X.temperatura_c - 23)**2 + rng.normal(0, .25, n))
display(X.assign(rendimiento_t_ha=y).head())


### 1.2. Simular registros de lluvia que faltan

Ocultamos la lluvia de **20 fincas**, elegidas al azar sin repetir. `NaN` significa dato ausente; no significa cero lluvia. Mantenemos su rendimiento conocido: pudo registrarse la cosecha aunque faltara la medición de lluvia.

La respuesta se creó **antes** de ocultar los datos para poder evaluar este caso didáctico. Más adelante completaremos las entradas ausentes con una estimación llamada **imputación**.

**Lee la salida:** debe haber 20 valores de lluvia sin registro. Las filas mostradas ayudan a reconocer cómo aparece un dato ausente en una tabla.


In [ ]:
X.loc[rng.choice(n, 20, replace=False), 'lluvia_mm'] = np.nan
print('Fincas sin registro de lluvia:', X.lluvia_mm.isna().sum())
display(X.loc[X.lluvia_mm.isna()].head(3))


### 1.3. Separar antes de aprender

Reservamos **180 fincas para entrenamiento** y **60 para prueba**. `test_size=.25` aparta el 25 %; `random_state=42` reproduce esa partición.

`X_tr` y `y_tr` son entradas y respuestas de entrenamiento. `X_te` y `y_te` son las de prueba. El modelo aprende con las primeras; sus predicciones se comprueban con las respuestas reservadas.

La mediana para completar vacíos y la escala también se aprenden solo con entrenamiento. Usar prueba para calcularlas introduciría información que debía permanecer reservada: **fuga de información**.

Aquí tratamos las fincas como independientes. Para mediciones repetidas de una finca o pronósticos por fecha, habría que separar por finca o respetar el tiempo. En este ejemplo no elegimos hiperparámetros; si comparáramos muchas configuraciones, necesitaríamos validación dentro de entrenamiento.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=.25, random_state=42
)
print('Fincas de entrenamiento:', len(X_tr))
print('Fincas de prueba:', len(X_te))


### 1.4. Crear una referencia sencilla

Antes de usar una regresión, necesitamos una respuesta básica contra la cual comparar. `DummyRegressor(strategy='mean')` aprende la **media del rendimiento de entrenamiento** y predice ese mismo valor para todas las fincas de prueba. No utiliza lluvia, temperatura ni fertilizante para distinguirlas.

`make_pipeline` encadena pasos. En esta referencia, `SimpleImputer(strategy='median')` completa entradas ausentes y después actúa el predictor de media. La imputación mantiene una preparación compatible, aunque este predictor no depende de las entradas.

`fit` aprende esos pasos únicamente con entrenamiento. **Lee la salida:** el número es la predicción de referencia, en t/ha; no es su error.


In [ ]:
base = make_pipeline(
    SimpleImputer(strategy='median'),
    DummyRegressor(strategy='mean')
)
base.fit(X_tr, y_tr)
print(f'La referencia predice {base.predict(X_te)[0]:.3f} t/ha para cada finca.')


In [ ]:
# La preparación aprende solo con entrenamiento; no usa las respuestas de prueba.
mediana_entrenamiento = X_tr['lluvia_mm'].median()
mediana_aprendida = base[0].statistics_[0]
assert np.isclose(mediana_aprendida, mediana_entrenamiento)
print('Mediana aprendida en entrenamiento:', round(mediana_aprendida, 3), 'mm')
print('Ausencias en entrenamiento:', X_tr.lluvia_mm.isna().sum())
print('Ausencias en prueba:', X_te.lluvia_mm.isna().sum())
preparacion_prueba = pd.DataFrame(base[0].transform(X_te), columns=X_te.columns, index=X_te.index)
assert not preparacion_prueba.isna().any().any()
display(pd.DataFrame({'lluvia_original_mm': X_te.lluvia_mm,
                      'lluvia_preparada_mm': preparacion_prueba.lluvia_mm}).head(8))
print('Completar una ausencia permite calcular; no recupera la medición que falta.')


### Lee, comprueba y explica

Veinte registros de lluvia están ausentes. Se reservan 60 fincas y la mediana se aprende únicamente con las 180 de entrenamiento.

**Prueba pequeña:** Imputar no recupera una medición real. La misma mediana aprendida se aplica después a la prueba.

**Vuelve al caso:** Aprende la regla de preparación con entrenamiento, aplícala a los otros conjuntos y conserva la diferencia entre una medición y una estimación.

**Pregunta:** ¿Por qué completar una ausencia y escalar una variable son operaciones diferentes?

**Respuesta razonada:** Imputar estima un valor faltante. Escalar cambia la representación numérica de valores disponibles. Ninguna operación debe aprender estadísticas de la prueba; tampoco convierte una estimación en una medida real.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-preparacion)


## Tema 3 · Predecir y comparar con una referencia

**Lo que vas a poder hacer:** Construir una comparación de estimaciones con errores en unidades útiles.

Estimar cosechas ayuda a planificar; necesitamos saber cuánto puede equivocarse la estimación.

### Entiende la idea

La regresión responde preguntas sobre cantidades, como rendimiento en t/ha. Una recta es una de sus posibles reglas: predicción = intercepto + pendiente × entrada. En la ilustración movemos sus parámetros manualmente; en el cuaderno se ajustan con datos. Después medimos cuánto se separan sus predicciones de los valores observados. El criterio usado para ajustar y la métrica usada para evaluar pueden ser diferentes.

### Un caso pequeño antes del programa

**La recta se equivoca en cada finca**

Seis fincas con datos inventados. Entrada: fertilizante en kg/ha; respuesta: rendimiento en t/ha. La regla inicial es predicción = 1 + 0,025 × fertilizante. La tabla y el explorador usan las mismas seis fincas.

| Finca | Fertilizante (kg/ha) | Observado (t/ha) | Predicho (t/ha) |
| --- | --- | --- | --- |
| A | 40 | 2,20 | 2,00 |
| B | 60 | 2,80 | 2,50 |
| C | 80 | 3,10 | 3,00 |
| D | 100 | 3,70 | 3,50 |
| E | 120 | 4,20 | 4,00 |
| F | 140 | 4,70 | 4,50 |

1. Para A calculamos 1 + 0,025 × 40 = 2,00 t/ha; su distancia a 2,20 es 0,20.
2. Los seis errores absolutos son 0,20; 0,30; 0,10; 0,20; 0,20 y 0,20. Suman 1,20 t/ha.
3. Dividimos entre seis: MAE = 0,20 t/ha. Es un promedio, no el error idéntico de cada finca.

**Qué demuestra:** La recta inicial tiene un error absoluto medio de 0,20 t/ha sobre estas seis fincas. Para evaluar su uso con otras fincas se necesita una prueba reservada. La pendiente describe la recta, no una dosis de fertilizante recomendada.

### Ahora identifica los datos de esta práctica

Compara referencia y regresión sobre las mismas 60 fincas. Las barras muestran error absoluto medio en t/ha; la última tabla permite revisar fincas individuales.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** El rendimiento observado se usa para comprobar; no se introduce como una entrada al predecir.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


### 1.1. Crear la tabla y su respuesta conocida

Generamos lluvia entre 200 y 800 mm, temperatura entre 16 y 30 °C y fertilizante entre 50 y 180 kg/ha. Son rangos elegidos para la simulación, no límites agronómicos generales.

El rendimiento **también se inventa** con una regla: aumenta con lluvia y fertilizante; disminuye al alejarse de 23 °C; lleva una pequeña variación aleatoria. Los números de esa regla son decisiones didácticas. Así conocemos una respuesta para cada fila y podemos comprobar después las predicciones.

`rng.uniform` genera las entradas dentro de los rangos. `rng.normal` añade variación. `**2` eleva al cuadrado la diferencia de temperatura: crea una relación curva. El modelo de regresión aprenderá de las filas; no recibe esta regla generadora.

**Lee la salida:** cada fila muestra las tres entradas y el rendimiento simulado correspondiente.


In [ ]:
n = 240
X = pd.DataFrame({
    'lluvia_mm': rng.uniform(200, 800, n),
    'temperatura_c': rng.uniform(16, 30, n),
    'fertilizante_kg_ha': rng.uniform(50, 180, n)
})
y = (1 + .004 * X.lluvia_mm + .015 * X.fertilizante_kg_ha
     - .03 * (X.temperatura_c - 23)**2 + rng.normal(0, .25, n))
display(X.assign(rendimiento_t_ha=y).head())


### 1.2. Simular registros de lluvia que faltan

Ocultamos la lluvia de **20 fincas**, elegidas al azar sin repetir. `NaN` significa dato ausente; no significa cero lluvia. Mantenemos su rendimiento conocido: pudo registrarse la cosecha aunque faltara la medición de lluvia.

La respuesta se creó **antes** de ocultar los datos para poder evaluar este caso didáctico. Más adelante completaremos las entradas ausentes con una estimación llamada **imputación**.

**Lee la salida:** debe haber 20 valores de lluvia sin registro. Las filas mostradas ayudan a reconocer cómo aparece un dato ausente en una tabla.


In [ ]:
X.loc[rng.choice(n, 20, replace=False), 'lluvia_mm'] = np.nan
print('Fincas sin registro de lluvia:', X.lluvia_mm.isna().sum())
display(X.loc[X.lluvia_mm.isna()].head(3))


### 1.3. Separar antes de aprender

Reservamos **180 fincas para entrenamiento** y **60 para prueba**. `test_size=.25` aparta el 25 %; `random_state=42` reproduce esa partición.

`X_tr` y `y_tr` son entradas y respuestas de entrenamiento. `X_te` y `y_te` son las de prueba. El modelo aprende con las primeras; sus predicciones se comprueban con las respuestas reservadas.

La mediana para completar vacíos y la escala también se aprenden solo con entrenamiento. Usar prueba para calcularlas introduciría información que debía permanecer reservada: **fuga de información**.

Aquí tratamos las fincas como independientes. Para mediciones repetidas de una finca o pronósticos por fecha, habría que separar por finca o respetar el tiempo. En este ejemplo no elegimos hiperparámetros; si comparáramos muchas configuraciones, necesitaríamos validación dentro de entrenamiento.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=.25, random_state=42
)
print('Fincas de entrenamiento:', len(X_tr))
print('Fincas de prueba:', len(X_te))


### 1.4. Crear una referencia sencilla

Antes de usar una regresión, necesitamos una respuesta básica contra la cual comparar. `DummyRegressor(strategy='mean')` aprende la **media del rendimiento de entrenamiento** y predice ese mismo valor para todas las fincas de prueba. No utiliza lluvia, temperatura ni fertilizante para distinguirlas.

`make_pipeline` encadena pasos. En esta referencia, `SimpleImputer(strategy='median')` completa entradas ausentes y después actúa el predictor de media. La imputación mantiene una preparación compatible, aunque este predictor no depende de las entradas.

`fit` aprende esos pasos únicamente con entrenamiento. **Lee la salida:** el número es la predicción de referencia, en t/ha; no es su error.


In [ ]:
base = make_pipeline(
    SimpleImputer(strategy='median'),
    DummyRegressor(strategy='mean')
)
base.fit(X_tr, y_tr)
print(f'La referencia predice {base.predict(X_te)[0]:.3f} t/ha para cada finca.')


### 1.5. Aprender una regresión con las tres entradas

Este segundo pipeline hace tres pasos en orden:

1. **Imputar:** completa una lluvia ausente con la mediana aprendida de las lluvias de entrenamiento.
2. **Estandarizar:** expresa cada entrada respecto a su media y su dispersión, aprendidas con entrenamiento. Así se manejan magnitudes diferentes; esto no añade información ni garantiza una mejora.
3. **Regresión lineal:** aprende un valor inicial y un peso para cada entrada para estimar rendimiento.

Para datos de prueba, el pipeline aplica las mismas reglas y los pesos ya aprendidos; no los recalcula. La relación de temperatura que inventamos es curva y esta regresión usa las tres variables sin potencias: puede quedar error por esa simplificación y por la variación aleatoria.

**Lee la salida:** la mediana está en mm y es la que usará para cualquier lluvia ausente. No tiene por qué ser 600 mm: ese número pertenecía al ejemplo ilustrativo de cuatro fincas de la presentación.


In [ ]:
lineal = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LinearRegression()
)
lineal.fit(X_tr, y_tr)
print(f'Mediana de lluvia aprendida: {lineal[0].statistics_[0]:.3f} mm')


### 1.6. Comparar predicciones con respuestas reservadas

`predict(X_te)` estima rendimientos para las 60 fincas de prueba. Después comparamos cada predicción con su rendimiento conocido, `y_te`. Llamamos **error absoluto** a la distancia entre ambos valores, sin signo; el **MAE** es el promedio de esas distancias: `MAE = promedio(|y − ŷ|)`, donde `ŷ` es la predicción.

**Cómo leer la tabla y el gráfico:**

- Ambos modelos se evalúan sobre las mismas 60 fincas. Cada barra muestra su MAE, no el rendimiento promedio.
- El eje vertical usa t/ha. Un MAE de 0,46 significa un error absoluto promedio cercano a 0,46 toneladas por hectárea en esta prueba; no que todas las fincas tengan ese error.
- La barra más baja corresponde al menor error medio. Compara la regresión con la referencia para saber si las entradas ayudaron en este ejemplo.
- La última tabla muestra cinco fincas de prueba: respuesta conocida, predicción de regresión y error absoluto de cada una.

**Alcance:** con una ejecución completa desde la semilla 42, la referencia obtiene aproximadamente **0,705 t/ha** y la regresión **0,459 t/ha**. La regresión mejora esta referencia en esta simulación. No demuestra que funcionará igual en fincas reales, ni que una variable cause cambios en el rendimiento. Las cifras pueden cambiar si alteras los datos o ejecutas celdas aleatorias de nuevo sin reiniciar el recorrido.


In [ ]:
pred_base = base.predict(X_te)
pred_lineal = lineal.predict(X_te)
resultados = pd.DataFrame({
    'modelo': ['Media de entrenamiento', 'Regresión lineal'],
    'MAE_t_ha': [mean_absolute_error(y_te, pred_base),
                 mean_absolute_error(y_te, pred_lineal)]
})
display(resultados)
plt.bar(resultados.modelo, resultados.MAE_t_ha)
plt.ylabel('MAE en prueba (t/ha)')
plt.title('Comparar con una referencia: mismas 60 fincas de prueba')
plt.show()
comparacion = pd.DataFrame({
    'rendimiento_conocido_t_ha': y_te,
    'prediccion_lineal_t_ha': pred_lineal,
    'error_absoluto_t_ha': np.abs(y_te - pred_lineal)
})
display(comparacion.head().round(3))


### Lectura del gráfico 1 · ¿Las entradas mejoraron la referencia?

**Eje horizontal:** los dos modelos. **Eje vertical:** error absoluto medio en las 60 fincas de prueba, expresado en t/ha. La altura de una barra representa error, no producción ni número de fincas. Cada barra resume 60 comparaciones entre respuesta conocida y predicción.

En esta ejecución, la referencia tiene **0,705 t/ha** de MAE y la regresión **0,459 t/ha**. La segunda barra es más baja: la regresión comete menos error medio sobre la misma prueba. Eso es lo que podemos concluir para este ejemplo.

En la última tabla, una finca tiene rendimiento conocido **4,338**, predicción **4,889** y error absoluto **0,551 t/ha**. La predicción quedó por encima de la respuesta; otra fila puede quedar por debajo. El MAE promedia las distancias sin compensar unos errores con otros. No es un porcentaje de aciertos ni el error de todas las fincas.


**Conexión con la presentación:** la diapositiva 6 resume aprender y predecir; aquí ves las filas, los pasos y el error que produce ese recorrido. La diapositiva 8 usa cuatro fincas ilustrativas para explicar la mediana; este laboratorio calcula su propia mediana con sus 180 fincas de entrenamiento.

Documentación de consulta: [imputación](https://scikit-learn.org/stable/modules/impute.html), [pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline), [MAE](https://scikit-learn.org/stable/modules/model_evaluation.html#mean-absolute-error).


### Lee, comprueba y explica

Compara referencia y regresión sobre las mismas 60 fincas. Las barras muestran error absoluto medio en t/ha; la última tabla permite revisar fincas individuales.

**Prueba pequeña:** El rendimiento observado se usa para comprobar; no se introduce como una entrada al predecir.

**Vuelve al caso:** El MAE describe el tamaño medio del error en las unidades de la respuesta. Para evaluar casos nuevos, calcúlalo con datos reservados.

**Pregunta:** ¿De dónde salen pendiente e intercepto y qué mide el MAE?

**Respuesta razonada:** En la demostración eliges pendiente e intercepto manualmente. En el entrenamiento los estima el algoritmo con los datos. El MAE resume el error absoluto de las predicciones en las unidades de la respuesta; no indica causalidad.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-regresion)


## Cierra el encuentro con una explicación

1. **Primero la pregunta, después el modelo:** Define la unidad de cada fila, las entradas disponibles al decidir y la respuesta que quieres estimar. Comprueba las predicciones con casos que quedaron fuera del ajuste.
2. **Un registro vacío no es cero:** Aprende la regla de preparación con entrenamiento, aplícala a los otros conjuntos y conserva la diferencia entre una medición y una estimación.
3. **Una predicción también tiene error:** El MAE describe el tamaño medio del error en las unidades de la respuesta. Para evaluar casos nuevos, calcúlalo con datos reservados.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
